# 1. Setup

3C v2 compares the original 3C, 3C + Blind Novel Feature Recomposition (BNFR), and 3C + BNFR + Outlier Exposure (OE) at one matched unfreeze depth. Set depth to the corresponding 3C depth before training.

The shared protocol limits development to the training combinations. A blind crop–disease cross-product can incidentally reproduce an official held-out identity even without reading the test list. The default runs only the original 3C control. Set ALLOW_BLIND_RECOMPOSITION to True only if the course/project owner explicitly confirms incidental synthetic matches are permitted. If prohibited, keep it False. Do not open official test metadata to filter the synthetic pool.

[COMO, IJCAI 2025, Sections 3.2–3.4](https://ijcai.org/proceedings/2025/0143.pdf) uses CLIP features, text compatibility, extra disentanglement/discriminability objectives and confidence-based weighting. This adaptation uses 3C's dual DINOv2 towers and feature fusion, auxiliary primitive supervision for outliers, and a fixed score mixture selected on pseudo-unseen validation. It applies the uniform-target OE objective to fitted seen-pair scores and uses no CLIP text targets, novel-pair classification loss or confidence gate.

In [ ]:
import gc, json, math, sys
from pathlib import Path
from datetime import datetime
import timm, torch
import pandas as pd
from torch import nn
from timm.layers import set_reentrant_ckpt
from torch.nn import functional as F
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from common import config as cfg
from common.dataset import get_loaders, get_pairs, get_transforms, _make_loader, get_final_test_loaders
from common.evaluate import evaluate, collect_scores, top_k_accuracy
MODEL = 'vit_base_patch14_dinov2.lvd142m'
MODEL_NAME, FAMILY = 'approach3c_v2_dino_blind_recomposition', 'C'
DEBUG = True
ALLOW_BLIND_RECOMPOSITION = False
HP = dict(backbone=MODEL, batch_size=8 if DEBUG else 16, epochs=1 if DEBUG else 20,
          unfreeze_depth=None, backbone_lr=1e-5,
          crop_lr=1e-4, disease_lr=3e-4, weight_decay=1e-4, label_smoothing=0.1,
          seed=cfg.SEED, norm='imagenet', hidden_dim=256, feature_dim=128,
          dino_revision='main', fusion_attention_heads=4,
          primitive_weight_start=0.75, primitive_weight_end=0.25,
          pseudo_pairs=10, primitive_mix=[0.0, 0.25, 0.5, 0.75],
          bnfr_weights=[0.25, 0.5], oe_weights=[0.1, 0.25])

EPOCHS = HP['epochs']
cfg.NUM_WORKERS = 0  # Exact ToTnki runtime setting; common files are unchanged.
cfg.set_seed(cfg.SEED)
RUN = MODEL_NAME + ('_debug' if DEBUG else '') + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT = cfg.CHECKPOINT_DIR / RUN
environment = dict(python=sys.version, torch=str(torch.__version__), timm=str(timm.__version__), device=str(cfg.DEVICE))
ABLATIONS = ("3c", "3c_bnfr", "3c_bnfr_oe")

# 2. Data

Use the shared split, samplers and transforms. For tuning, hold out entire combinations from the training partition. Score their real images from the shared validation partition. Keep both primitives of each pseudo-unseen pair represented in the fit partition. Exclude all original training pairs from synthetic novel targets, including the pseudo-unseen pairs.

In [ ]:
pairs = get_pairs()

def common_loaders():
    cfg.set_seed(cfg.SEED)
    train, val, _, _ = get_loaders(HP['batch_size'], norm=HP['norm'], debug=DEBUG)
    train.pin_memory = val.pin_memory = False
    return train, val


def pseudo_loaders():
    train, val = common_loaders()
    fit_frame, val_frame = train.dataset.frame, val.dataset.frame
    active = set(fit_frame.pair_row)
    options = sorted(active & set(val_frame.pair_row))
    generator = torch.Generator().manual_seed(cfg.SEED)
    held_out = []
    for index in torch.randperm(len(options), generator=generator).tolist():
        row = options[index]
        remaining = pairs[sorted(active - {row})]
        crop, disease = pairs[row]
        if (remaining[:, 0] == crop).any() and (remaining[:, 1] == disease).any():
            active.remove(row)
            held_out.append(row)
        if len(held_out) == HP['pseudo_pairs']:
            break
    if not held_out:
        raise ValueError('No eligible pseudo-unseen pairs with both primitives in the fit split.')
    keep = sorted(active)
    fit = fit_frame[fit_frame.pair_row.isin(keep)].copy()
    fit['pair_row'] = fit.pair_row.map({row: i for i, row in enumerate(keep)})
    pseudo = val_frame[val_frame.pair_row.isin(held_out)].copy()
    train_transform, eval_transform = get_transforms(HP['norm'])
    fit_loader = _make_loader(fit, train_transform, HP['norm'], HP['batch_size'], True)
    pseudo_loader = _make_loader(pseudo, eval_transform, HP['norm'], HP['batch_size'], False)
    fit_loader.pin_memory = pseudo_loader.pin_memory = False
    return fit_loader, pseudo_loader, pairs[keep], held_out

# 3. Model

Keep 3C's visual towers, residual fusion and auxiliary classifiers. BNFR supervises the source crop and source disease of each novel feature pair. OE pushes the joint scorer toward uniform probabilities over fitting compositions. At inference the same scoring rule is applied to every candidate in order, without true or predicted membership labels.

In [ ]:
def snapshot(repository,revision,allow_download,patterns):
    from huggingface_hub import snapshot_download
    path=Path(snapshot_download(repository,revision=revision,allow_patterns=patterns,local_files_only=not allow_download))
    return path

class Dino(nn.Module):
    """Exact two-independent-tower initialization from ToTnki's released runner."""
    def __init__(self,config,device,allow_download=False,revision=None):
        super().__init__()
        import timm
        path=snapshot('timm/vit_base_patch14_dinov2.lvd142m',revision or config['dino_revision'],allow_download,
                      ['config.json','model.safetensors','pytorch_model.bin'])
        weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        def pinned_source_model(classes):
            # Same source factory, with only an explicit weight-file binding added.
            model=timm.create_model(MODEL,pretrained=True,num_classes=classes,img_size=cfg.IMAGE_SIZE,
                                    pretrained_cfg_overlay={'file':str(weights)})
            model.requires_grad_(False);model.head.requires_grad_(True)
            return model
        self.crop=pinned_source_model(cfg.NUM_CROPS)
        self.disease=pinned_source_model(cfg.NUM_DISEASES)
        self.augmentation_rng_state=torch.get_rng_state().clone()
        self.width=self.crop.num_features
        self.metadata=dict(repository='timm/vit_base_patch14_dinov2.lvd142m',revision=path.name,
            architecture=MODEL,width=self.width,independent_visual_towers=2,
            initialization='ToTnki source factory settings, crop then disease, explicitly pinned public weight file',
            preprocessing='common imagenet; img_size224; timm native position resampling and pre_logits')
        self.to(device)
    def policy(self, depth):
        self.requires_grad_(False)
        for tower in (self.crop, self.disease):
            tower.blocks[-depth:].requires_grad_(True)
            tower.set_grad_checkpointing(True)

    def features(self,pixels):
        return (self.crop.forward_head(self.crop.forward_features(pixels),pre_logits=True).float(),
                self.disease.forward_head(self.disease.forward_features(pixels),pre_logits=True).float())
    def forward(self,pixels):return self.features(pixels)

def projection(width,hidden,out):return nn.Sequential(nn.Linear(width,hidden),nn.ReLU(),nn.Linear(hidden,out))




class Model(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        width, hidden, dim = backbone.width, HP['hidden_dim'], HP['feature_dim']
        self.crop_projection = projection(width, hidden, dim)
        self.disease_projection = projection(width, hidden, dim)
        self.crop_aux = nn.Linear(dim, cfg.NUM_CROPS)
        self.disease_aux = nn.Linear(dim, cfg.NUM_DISEASES)
        self.attention = nn.MultiheadAttention(dim, HP['fusion_attention_heads'], batch_first=True)
        self.image_fusion = nn.Sequential(nn.LayerNorm(2 * dim), nn.Linear(2 * dim, dim), nn.GELU())
        self.crop_embedding = nn.Embedding(cfg.NUM_CROPS, dim)
        self.disease_embedding = nn.Embedding(cfg.NUM_DISEASES, dim)
        self.pair_fusion = nn.Sequential(nn.Linear(2 * dim, dim), nn.GELU())

    def features(self, images):
        crop, disease = self.backbone(images)
        return self.crop_projection(crop), self.disease_projection(disease)

    def score(self, crop, disease, candidate_pairs):
        combined = crop + disease
        tokens = torch.stack((crop + combined, disease + combined), dim=1)
        attended, _ = self.attention(tokens, tokens, tokens, need_weights=False)
        residuals = torch.stack((disease, crop), dim=1)
        image = self.image_fusion((attended + residuals).flatten(1))
        crop_ids, disease_ids = candidate_pairs.unbind(1)
        candidates = self.pair_fusion(torch.cat(
            (self.crop_embedding(crop_ids), self.disease_embedding(disease_ids)), dim=1))
        return image @ candidates.T


def sample_observed_pairs(crops, diseases, pair_rows, generator):
    """Sample at most one batch's worth of admissible cross-image combinations."""
    rows = pair_rows[crops[:, None], diseases[None, :]]
    eligible = (rows >= 0) & ~torch.eye(len(crops), dtype=torch.bool, device=rows.device)
    crop_sources, disease_sources = eligible.nonzero(as_tuple=True)
    order = torch.randperm(len(crop_sources), generator=generator)[:len(crops)].to(rows.device)
    crop_sources, disease_sources = crop_sources[order], disease_sources[order]
    return crop_sources, disease_sources, rows[crop_sources, disease_sources]


def training_loss(model, images, labels, candidates, pair_rows, generator, primitive_weight,
                  settings, observed, novel_generator):
    crop, disease = model.features(images)
    crops, diseases = candidates[labels].unbind(1)
    smoothing = HP['label_smoothing']
    primitive = (F.cross_entropy(model.crop_aux(crop), crops, label_smoothing=smoothing)
                 + F.cross_entropy(model.disease_aux(disease), diseases, label_smoothing=smoothing)) / 2
    joint = F.cross_entropy(model.score(crop, disease, candidates), labels, label_smoothing=smoothing)
    src_crop, src_disease, targets = sample_observed_pairs(crops, diseases, pair_rows, generator)
    if targets.numel():
        synthetic = model.score(crop[src_crop], disease[src_disease], candidates)
        joint = (joint + F.cross_entropy(synthetic, targets, label_smoothing=smoothing)) / 2
    loss = primitive_weight * primitive + (1 - primitive_weight) * joint
    novel_count = 0
    if settings['bnfr']:
        extra, novel_count = outlier_loss(model, crop, disease, crops, diseases, candidates,
                                          observed, novel_generator, settings)
        loss = loss + extra
    return loss, novel_count


def make_model(depth, metadata=None):
    cfg.set_seed(cfg.SEED)
    metadata = metadata or {}
    backbone = Dino(HP, cfg.DEVICE, allow_download=True, revision=metadata.get('dino', {}).get('revision'))
    backbone.policy(depth)
    cfg.set_seed(cfg.SEED)
    model = Model(backbone).to(cfg.DEVICE)
    torch.set_rng_state(backbone.augmentation_rng_state)
    return model, dict(dino=backbone.metadata)



def sample_novel(crops, diseases, observed, generator):
    eligible = ~observed[crops[:, None], diseases[None, :]]
    eligible.fill_diagonal_(False)
    left, right = eligible.nonzero(as_tuple=True)
    sample = torch.randperm(len(left), generator=generator)[:len(crops)].to(left.device)
    return left[sample], right[sample]


def uniform_oe(logits):
    return -F.log_softmax(logits, dim=1).mean() - math.log(logits.shape[1])


def outlier_loss(model, crop, disease, crops, diseases, candidates, observed, generator, settings):
    left, right = sample_novel(crops, diseases, observed, generator)
    if not left.numel():
        return crop.sum() * 0, 0
    novel_crop, novel_disease = crop[left], disease[right]
    primitive = (F.cross_entropy(model.crop_aux(novel_crop), crops[left])
                 + F.cross_entropy(model.disease_aux(novel_disease), diseases[right])) / 2
    loss = settings['bnfr'] * primitive
    if settings['oe']:
        loss = loss + settings['oe'] * uniform_oe(model.score(novel_crop, novel_disease, candidates))
    return loss, left.numel()


def make_predict(model, candidate_pairs, mix=0.0):
    candidates = candidate_pairs.to(cfg.DEVICE)
    crop_ids, disease_ids = candidates.unbind(1)

    @torch.no_grad()
    def predict(images):
        model.eval()
        crop, disease = model.features(images.to(cfg.DEVICE))
        joint = model.score(crop, disease, candidates)
        if mix == 0:
            return joint
        primitive = (model.crop_aux(crop).log_softmax(1)[:, crop_ids]
                     + model.disease_aux(disease).log_softmax(1)[:, disease_ids])
        return (1 - mix) * joint.log_softmax(1) + mix * primitive.log_softmax(1)
    return predict

# 4. Training

When enabled, tune BNFR strength, OE strength and score mixture using pseudo-unseen Top-1 only. Choose each configuration at the end of the same training budget, then start again from the same pretrained revisions and refit on the full shared train split. The control uses the original 3C loss and scorer. All refits save the last epoch to keep the compared epoch budget fixed; the original 3C notebook instead selects its best regular validation epoch.

In [ ]:
def fit_model(depth, train_loader, candidates, settings, revisions):
    if settings['bnfr'] and not ALLOW_BLIND_RECOMPOSITION:
        raise ValueError('Blind novel recomposition is disabled by the protocol gate.')
    cfg.set_seed(cfg.SEED)
    model, metadata = make_model(depth, revisions)
    candidates = candidates.to(cfg.DEVICE)
    pair_rows = torch.full((cfg.NUM_CROPS, cfg.NUM_DISEASES), -1, device=cfg.DEVICE)
    pair_rows[candidates[:, 0], candidates[:, 1]] = torch.arange(len(candidates), device=cfg.DEVICE)
    observed = torch.zeros_like(pair_rows, dtype=torch.bool)
    observed[pairs[:, 0], pairs[:, 1]] = True
    generator = torch.Generator().manual_seed(cfg.SEED)
    novel_generator = torch.Generator().manual_seed(cfg.SEED)
    groups = {'backbone': [], 'crop': [], 'disease': []}
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            key = 'backbone' if name.startswith('backbone.') else 'disease' if 'disease' in name else 'crop'
            groups[key].append(parameter)
    optimizer = torch.optim.AdamW(
        [dict(params=groups[key], lr=HP[key + '_lr']) for key in groups],
        weight_decay=HP['weight_decay'])
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    for epoch in range(1, EPOCHS + 1):
        model.train()
        model.backbone.eval()
        for tower in (model.backbone.crop, model.backbone.disease):
            tower.blocks[-depth:].train()
        progress = (epoch - 1) / (EPOCHS - 1) if EPOCHS > 1 else 0.5
        primitive_weight = HP['primitive_weight_start'] + progress * (
            HP['primitive_weight_end'] - HP['primitive_weight_start'])
        total_loss = synthetic_count = samples = 0
        for images, _, _, labels in train_loader:
            optimizer.zero_grad(set_to_none=True)
            value, synthetic = training_loss(model, images.to(cfg.DEVICE), labels.to(cfg.DEVICE),
                                             candidates, pair_rows, generator, primitive_weight,
                                             settings, observed, novel_generator)
            if not torch.isfinite(value):
                raise RuntimeError('Nonfinite training loss.')
            value.backward()
            optimizer.step()
            samples += len(labels)
            total_loss += value.item() * len(labels)
            synthetic_count += synthetic
        schedule.step()
        print(f"epoch {epoch}: loss={total_loss / samples:.4f}, novel examples={synthetic_count}")
    return model, metadata


def tune(name, depth, revisions):
    best = None
    bnfr_weights = HP['bnfr_weights'] if name != '3c' else [0.]
    oe_weights = HP['oe_weights'] if name == '3c_bnfr_oe' else [0.]
    for bnfr in bnfr_weights:
        for oe in oe_weights:
            fit, pseudo, fit_pairs, held_out = pseudo_loaders()
            model, revisions = fit_model(depth, fit, fit_pairs, dict(bnfr=bnfr, oe=oe), revisions)
            for mix in (HP['primitive_mix'] if bnfr else [0.]):
                scores, labels = collect_scores(make_predict(model, pairs, mix), pseudo)
                trial = dict(bnfr=bnfr, oe=oe, mix=mix, pseudo_top1=top_k_accuracy(scores, labels, 1))
                if best is None or trial['pseudo_top1'] > best['pseudo_top1']:
                    best = trial
            del model
    return dict(best, pseudo_pair_rows=held_out), revisions


depth = HP['unfreeze_depth']
if type(depth) is not int or depth not in (2, 4, 6, 8):
    raise ValueError('Set unfreeze_depth to the corresponding 3C depth (2, 4, 6, or 8).')
OUT.mkdir(parents=True, exist_ok=False)
set_reentrant_ckpt(False)
identity = dict(hp=HP, debug=DEBUG, model_name=MODEL_NAME, allow_blind=ALLOW_BLIND_RECOMPOSITION)
selected = {}
revisions = None
for name in (ABLATIONS if ALLOW_BLIND_RECOMPOSITION else ABLATIONS[:1]):
    if ALLOW_BLIND_RECOMPOSITION:
        settings, revisions = tune(name, depth, revisions)
    else:
        settings = dict(bnfr=0., oe=0., mix=0.)
    train, _ = common_loaders()
    model, revisions = fit_model(depth, train, pairs, settings, revisions)
    torch.save(dict(identity=identity, variant=name, settings=settings, metadata=revisions,
                    pairs=pairs, state_dict=model.state_dict()), OUT / f'{name}.pt')
    selected[name] = settings
    del model
    gc.collect()
(OUT / 'selection.json').write_text(json.dumps(dict(identity=identity, variants=selected), indent=2),
                                    encoding='utf8')
display(pd.DataFrame.from_dict(selected, orient='index'))

# 5. Evaluation

After configuration selection and refitting, opt in to official evaluation. The common evaluator reports seen/unseen Top-1 and Top-5, harmonic mean, seen confusion and robustness for each enabled ablation.

In [ ]:
RUN_FINAL_EVALUATION = False
reports = []
if RUN_FINAL_EVALUATION:
    if DEBUG:
        raise ValueError('Official evaluation requires a full non-DEBUG run.')
    selection = json.loads((OUT / 'selection.json').read_text(encoding='utf8'))
    expected = dict(hp=HP, debug=False, model_name=MODEL_NAME, allow_blind=ALLOW_BLIND_RECOMPOSITION)
    variants = ABLATIONS if ALLOW_BLIND_RECOMPOSITION else ABLATIONS[:1]
    if selection['identity'] != expected or tuple(selection['variants']) != variants:
        raise ValueError('Restore the frozen settings for this run.')
    RESULTS = cfg.RESULTS_DIR / OUT.name
    RESULTS.mkdir(parents=True, exist_ok=False)
    previous_results = cfg.RESULTS_DIR
    try:
        cfg.RESULTS_DIR = RESULTS
        for name in variants:
            checkpoint = torch.load(OUT / f'{name}.pt', map_location='cpu', weights_only=True)
            settings = selection['variants'][name]
            if (checkpoint['identity'] != expected or checkpoint['variant'] != name
                    or checkpoint['settings'] != settings or not torch.equal(checkpoint['pairs'], pairs)):
                raise ValueError('Checkpoint does not match the frozen selection.')
            model, metadata = make_model(HP['unfreeze_depth'], checkpoint['metadata'])
            if metadata != checkpoint['metadata']:
                raise ValueError('Public pretrained model identity changed.')
            model.load_state_dict(checkpoint['state_dict'])
            model.eval().requires_grad_(False)
            for tower in (model.backbone.crop, model.backbone.disease):
                tower.set_grad_checkpointing(False)
            final_pairs = get_pairs(final_evaluation=True)
            seen, unseen = get_final_test_loaders(
                HP['batch_size'], norm=HP['norm'], final_evaluation=True)
            predict = make_predict(model, final_pairs, settings['mix'])
            report = evaluate(predict, seen, unseen, f'{MODEL_NAME}_{name}',
                              dict(HP, variant=name, selection=settings), robust=True)
            reports.append(report)
            del model, predict, checkpoint
    finally:
        cfg.RESULTS_DIR = previous_results

# 6. Save

Keep the checkpoints and selection manifest. The shared evaluator writes individual JSON reports and seen confusion files; this section writes a comparison table.

In [ ]:
if RUN_FINAL_EVALUATION:
    table = pd.DataFrame(reports)[['model', 'seen_top1', 'seen_top5',
                                   'unseen_top1', 'unseen_top5', 'harmonic_mean']]
    table.to_csv(RESULTS / 'comparison.csv', index=False)
    display(table)
else:
    print('Checkpoints and selection:', OUT)